# 📘 Day 37 — Feature Importance Analysis

**Author:** Sahil-K-Y 
**Phase:** 3 — Tree Models & SVM 
**Date:** Day 037 of 214-Day AI/ML Roadmap

---
## 📌 Overview

Understanding **which features matter most** is critical for model interpretability, feature selection, and debugging. Tree-based models offer built-in importance measures.

### 🎯 Learning Objectives
- Compute **Impurity-based (MDI) importance** from tree models
- Compute **Permutation importance** (model-agnostic)
- Understand the **bias in impurity-based importance** (high-cardinality trap)
- Use feature importance for **feature selection**
- Visualize importances with **horizontal bar plots**

---
## 🧠 1. Impurity-Based Importance (MDI)

### How it works:
- For each feature, sum the **decrease in impurity** (Gini/entropy) across ALL splits that use it
- Weighted by the number of samples reaching each split
- Averaged across all trees in the forest

### Access in sklearn:
```python
rf = RandomForestClassifier(n_estimators=100)
rf.fit(X_train, y_train)
importances = rf.feature_importances_  # array of shape (n_features,)
```

### ⚠️ Known Bias:
- MDI importance is **biased toward high-cardinality features** (many unique values)
- Continuous features get higher importance than binary features, even if less predictive
- Random noise features with many unique values may appear "important"
- **Solution:** Use permutation importance as a complement

---
## 🧠 2. Permutation Importance

### How it works:
1. Train model, record **baseline score** on validation set
2. For each feature:
   - **Shuffle** that feature's values randomly
   - Record the **drop in score**
   - Bigger drop = more important feature
3. Repeat multiple times for stability

### Access in sklearn:
```python
from sklearn.inspection import permutation_importance
result = permutation_importance(rf, X_test, y_test, n_repeats=10, random_state=42)
result.importances_mean  # mean importance
result.importances_std   # std across repeats
```

### Advantages over MDI:
- **Not biased** toward high-cardinality features
- **Model-agnostic** — works with any model (SVM, KNN, etc.)
- Measured on **validation data** — reflects actual predictive power
- Can detect features that are **important together** (interaction effects)

---
## 🧠 3. MDI vs Permutation Comparison

| Aspect | MDI (Impurity-based) | Permutation |
|--------|---------------------|-------------|
| **Speed** | Fast (computed during training) | Slower (re-evaluate model) |
| **Bias** | Biased toward high-cardinality | Unbiased |
| **Data** | Training data | Validation/test data |
| **Correlation** | Splits importance among correlated features | Affected by correlated features |
| **Model** | Tree-based only | Any model |
| **Negative values** | Never | Possible (feature hurts model) |

---
## 🧠 4. Feature Selection Using Importance

### Strategy: SelectFromModel
```python
from sklearn.feature_selection import SelectFromModel
selector = SelectFromModel(rf, threshold='median')
X_selected = selector.fit_transform(X_train, y_train)
```

### Manual Threshold:
1. Compute importances
2. Sort features by importance
3. Cumulatively add features until reaching target accuracy
4. Drop features that add <0.5% accuracy improvement

### Benefits of Feature Selection:
- **Faster** training and prediction
- **Less overfitting** (fewer dimensions)
- **Better interpretability** (fewer features to explain)
- **Lower data collection costs** in production

---
## 💻 Imports & Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.inspection import permutation_importance
from sklearn.feature_selection import SelectFromModel
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import LabelEncoder, OrdinalEncoder
from sklearn.metrics import accuracy_score, classification_report
from sklearn.datasets import fetch_openml

import warnings
warnings.filterwarnings('ignore')

plt.style.use('ggplot')
sns.set_palette('Set2')
print('✅ All libraries imported successfully!')

---
## 📊 Dataset: Adult Income (Census)

The **Adult/Census Income** dataset predicts whether a person earns **>$50K/year** based on demographic and employment features.

**48,842 samples, 14 features** — mix of numerical and categorical.

Features: age, workclass, education, education-num, marital-status, occupation, relationship, race, sex, capital-gain, capital-loss, hours-per-week, native-country

**Why this dataset?** A **feature-rich dataset with mixed types** where feature importance analysis reveals surprising insights about which demographic factors truly predict income.

In [ ]:
# --- Load Adult Income Dataset ---
adult = fetch_openml('adult', version=2, as_frame=True, parser='auto')
df = adult.frame.dropna().reset_index(drop=True)

# Encode categoricals
cat_cols = df.select_dtypes(include='category').columns.tolist()
num_cols = df.select_dtypes(include='number').columns.tolist()

target_col = adult.target_names[0] if adult.target_names else 'class'
if target_col in cat_cols:
    cat_cols.remove(target_col)
if target_col in num_cols:
    num_cols.remove(target_col)

# Encode target
le_target = LabelEncoder()
y = le_target.fit_transform(df[target_col])

# Encode features
df_encoded = df.drop(columns=[target_col]).copy()
for col in cat_cols:
    df_encoded[col] = OrdinalEncoder().fit_transform(df_encoded[[col]])

X = df_encoded.astype(float)

print(f'Shape: {X.shape}')
print(f'Target: {le_target.classes_}')
print(f'Target distribution: {dict(zip(*np.unique(y, return_counts=True)))}')
print(f'\nNumerical features: {num_cols}')
print(f'Categorical features: {cat_cols}')
df.head()

---
## ✏️ Exercise 1: MDI Importance
Compute impurity-based importance.

**Tasks:**
1. Train RF (n_estimators=200) on training set
2. Extract feature_importances_
3. Create horizontal bar plot (sorted, top 15)
4. Which features are most important?

In [ ]:
# YOUR CODE HERE


---
## ✏️ Exercise 2: Permutation Importance
Compute permutation importance on test set.

**Tasks:**
1. Use `permutation_importance` with n_repeats=10
2. Plot mean importance with error bars (std)
3. Compare ranking with MDI importance
4. Any features that changed rank significantly?

In [ ]:
# YOUR CODE HERE


---
## ✏️ Exercise 3: MDI vs Permutation Side-by-Side
Visual comparison of both methods.

**Tasks:**
1. Create side-by-side bar plots (MDI left, Permutation right)
2. Highlight features that rank differently
3. Create a rank comparison table
4. Discuss disagreements and which to trust

In [ ]:
# YOUR CODE HERE


---
## ✏️ Exercise 4: Feature Selection with Importance
Use importance to select features and measure impact.

**Tasks:**
1. Use SelectFromModel with RF (threshold='median')
2. Compare accuracy: all features vs selected features
3. Try keeping only top 5, top 8, top 10 features
4. Plot #features vs accuracy to find minimum needed

In [ ]:
# YOUR CODE HERE


---
## ✏️ Exercise 5: Random Noise Feature Test
Prove MDI bias with a random noise column.

**Tasks:**
1. Add a random noise column (high cardinality: np.random.rand)
2. Add a random integer column (low cardinality: randint 0-5)
3. Train RF and check MDI importance of both noise columns
4. Check permutation importance of both
5. Which method correctly identifies them as unimportant?

In [ ]:
# YOUR CODE HERE


---
## 📝 Key Takeaways
1. Top 3 most important features: ...
2. MDI vs Permutation agreement: ...
3. Minimum features needed for 95% accuracy: ...
4. Random noise MDI bias confirmed: ...